# Chapters 9–11 — Temporal LSTM, Attention, and joint outputs

**** The research used consecutive synchronized frames, per-backbone Flatten, feature Concatenation, LSTM → Attention → Dense, and simultaneous steering/speed prediction. **[implementation]** The exact attention variant, LSTM size, and previous window length are unknown. This notebook demonstrates an additive attention variant using synthetic **feature vectors**, not DBNet frames.

**Dataset attribution:** When training on DBNet, cite Chen et al. (2018), *LiDAR-Video Driving Dataset: Learning Driving Policies Effectively*, CVPR. Original project: https://github.com/driving-behavior/DBNet. Full BibTeX: ../CITATION_DBNet.bib.

In [ ]:
%pip install torch
import torch
from autonomous_driving.temporal_model import build_temporal_model

# Example dimensions are implementation demonstrations, example hyperparameters.
BATCH = 2
SEQUENCE_LENGTH = 4
FUSED_FEATURES = 24
HIDDEN = 16
model = build_temporal_model(FUSED_FEATURES, hidden_size=HIDDEN)
synthetic_fused = torch.randn(BATCH, SEQUENCE_LENGTH, FUSED_FEATURES)
outputs = model(synthetic_fused)
for name, value in outputs.items():
    print(name, tuple(value.shape))
assert outputs['steering_deg'].shape == (BATCH,)
assert outputs['speed_kmh'].shape == (BATCH,)

## Interpretability check

Attention weights are computed over the sequence and sum to one for each sample. These are not proof of causal explanation. Both predictions are generated from a common context vector. The model does not insert a Dense layer before the LSTM.

In [ ]:
print('Attention weights:', outputs['attention_weights'].detach())
print('Row sums:', outputs['attention_weights'].sum(dim=1).detach())
assert torch.allclose(outputs['attention_weights'].sum(dim=1), torch.ones(BATCH), atol=1e-6)

## Differentiability and loss demonstration

**[implementation]** A demonstration loss below is not the verified previous training loss. The research describes RMSE but the joint target aggregation is unresolved. Real targets require correct steering/speed units and a documented scaling policy.

In [ ]:
targets = torch.zeros(BATCH, 2)  # synthetic labels only
pred = torch.stack([outputs['steering_deg'], outputs['speed_kmh']], dim=1)
loss = torch.sqrt(torch.mean((pred - targets) ** 2) + 1e-8)
loss.backward()
print('Synthetic demonstration RMSE:', float(loss.detach()))

## Integrating real backbones

Use `stack_temporal_features(camera_model, lidar_model, camera_frames, lidar_frames)` with aligned `[B,T,3,299,299]` tensors. Pass the resulting `[B,T,F]` into `build_temporal_model(input_size=F)`. **Caution:** unpooled Flatten can create very large feature vectors and LSTM parameter matrices. Record actual tensor dimensions and GPU memory before training. The original sequence length and exact attention formulation remain unknown.